# **Guidelines for Prompting**
In this lesson, you'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.

## Setup


#### Create a .env file with your OpenAI API key

Create a `.env` file in the root of your project and add your OpenAI API key:

```
OPENAI_API_KEY=your_api_key_here
```

The `.env` file stores configuration values (such as API keys) outside of your source code. The `python-dotenv` library loads these values into environment variables at runtime, allowing your code to access them securely with `os.getenv()` without hardcoding sensitive information.


> ⚠️ WARNING: 
> - Never commit your `.env` file to version control, as it contains your private API key
> - If you're using Git, create (or open) a file named `.gitignore` in the root of your project and add the following line:
> 
>   ```
>   # your .gitignore file should include the following line:
>   
>   .env
>   
>   ```
> 
> This tells Git to ignore the `.env` file so it won't be included in commits or uploaded to repositories such as GitHub. This helps prevent accidentally exposing your API key!
> 

<br>

#### Load the API key and relevant Python libaries.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

<br>

#### Helper function


For this demo, we will use OpenAI's `gpt-5.4-nano` model and the [responses endpoint](https://platform.openai.com/docs/api-reference/responses). 

This helper function will make it easier to use prompts and look at the generated outputs:

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-5.4-nano"):
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.output_text


## Prompting Principles
- **Principle 1: Write clear and specific instructions**
- **Principle 2: Give the model time to “think”**

### Tactics

#### Tactic 1: Use delimiters to clearly indicate distinct parts of the input
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`

In [3]:
text = f"""
You should express what you want a model to do by \ 
providing instructions that are as clear and \ 
specific as you can possibly make them. \ 
This will guide the model towards the desired output, \ 
and reduce the chances of receiving irrelevant \ 
or incorrect responses. Don't confuse writing a \ 
clear prompt with writing a short prompt. \ 
In many cases, longer prompts provide more clarity \ 
and context for the model, which can lead to \ 
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \ 
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

Provide clear, specific, and often longer instructions to guide the model toward the desired output and reduce irrelevant or incorrect responses.


#### Tactic 2: Ask for a structured output
- JSON, HTML

In [4]:
prompt = f"""
Generate a list of three made-up book titles along \ 
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

```json
[
  {
    "book_id": "B001",
    "title": "The Clockwork Orchard",
    "author": "Marin Ellery",
    "genre": "Steampunk Fantasy"
  },
  {
    "book_id": "B002",
    "title": "Beneath the Neon Tide",
    "author": "Jasper Kwon",
    "genre": "Cyberpunk Thriller"
  },
  {
    "book_id": "B003",
    "title": "Letters from a Silent Planet",
    "author": "Dr. Celeste Navarro",
    "genre": "Science Fiction"
  }
]
```


#### Tactic 3: Ask the model to check whether conditions are satisfied

In [5]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some \ 
water boiling. While that's happening, \ 
grab a cup and put a tea bag in it. Once the water is \ 
hot enough, just pour it over the tea bag. \ 
Let it sit for a bit so the tea can steep. After a \ 
few minutes, take out the tea bag. If you \ 
like, you can add some sugar or milk to taste. \ 
And that's it! You've got yourself a delicious \ 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.  
Step 2 - Grab a cup and put a tea bag in it.  
Step 3 - Pour the hot water over the tea bag.  
Step 4 - Let the tea steep for a bit.  
Step 5 - After a few minutes, take out the tea bag.  
Step 6 - Add sugar or milk to taste (optional).  
Step 7 - Enjoy your cup of tea.


In [6]:
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \ 
walk in the park. The flowers are blooming, and the \ 
trees are swaying gently in the breeze. People \ 
are out and about, enjoying the lovely weather. \ 
Some are having picnics, while others are playing \ 
games or simply relaxing on the grass. It's a \ 
perfect day to spend time outdoors and appreciate the \ 
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


#### Tactic 4: "Few-shot" prompting

In [7]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \ 
valley flows from a modest spring; the \ 
grandest symphony originates from a single note; \ 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

<grandparent>: Resilience is the quiet strength that keeps going when the path grows rough; it bends like a young tree in the wind, yet does not break. It is learning to stand back up, again and again, with a little more wisdom each time. Like a stone worn smooth by steady water, you don’t become stronger all at once—you become stronger by continuing.


### Principle 2: Give the model time to “think” 

#### Tactic 1: Specify the steps required to complete a task

In [8]:
text = f"""
In a charming village, siblings Jack and Jill set out on \ 
a quest to fetch water from a hilltop \ 
well. As they climbed, singing joyfully, misfortune \ 
struck—Jack tripped on a stone and tumbled \ 
down the hill, with Jill following suit. \ 
Though slightly battered, the pair returned home to \ 
comforting embraces. Despite the mishap, \ 
their adventurous spirits remained undimmed, and they \ 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
Jack and Jill, siblings in a charming village, go to fetch water from a hilltop well, suffer a mishap when Jack falls and Jill follows, then return home battered but still adventurous.  
Jack et Jill, frères et sœurs dans un charmant village, partent chercher de l’eau dans un puits au sommet d’une colline, subissent un accident lorsque Jack tombe et Jill le suit, puis rentrent chez eux meurtris mais toujours aventureux.  
Jack, Jill  
{"french_summary":"Jack et Jill, frères et sœurs dans un charmant village, partent chercher de l’eau dans un puits au sommet d’une colline, subissent un accident lorsque Jack tombe et Jill le suit, puis rentrent chez eux meurtris mais toujours aventureux.","num_names":2}


#### Ask for output in a specified format

In [9]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in Italian summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Summary: In a charming village, siblings Jack and Jill go to fetch water, but Jack trips and both tumble down the hill, then they return home hurt yet comforted and keep exploring.  
Translation: Dans un charmant village, les frères et sœurs Jack et Jill partent chercher de l’eau, mais Jack trébuche et tous deux dévalent la colline, puis ils rentrent chez eux blessés mais réconfortés et continuent d’explorer.  
Names: Jack, Jill  
Output JSON:  
```json
{
  "french_summary": "Dans un charmant village, les frères et sœurs Jack et Jill partent chercher de l’eau, mais Jack trébuche et tous deux dévalent la colline, puis ils rentrent chez eux blessés mais réconfortés et continuent d’explorer.",
  "num_names": 2
}
```


## Model Limitations: Hallucinations

> ℹ️ Hallucinations:
>
> A hallucination is when an AI model generates information that sounds plausible and confident but is incorrect, misleading, or entirely fabricated. Hallucinations can include made-up facts, incorrect citations, nonexistent code, or inaccurate explanations. Because language models generate text by predicting likely sequences of words rather than retrieving verified facts, their responses should be treated as outputs to evaluate, not as guaranteed truth.

In [ ]:
# note: Boie is a real company, the product name is not real.

prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

I can help, but I need one quick clarification: **do you mean “Boie” as in the brand “BOIE” (often sold via Amazon/retailers), and is the product called “AeroGlide UltraSlim Smart Toothbrush”?** Product listings sometimes vary by region and seller, and I don’t want to give you incorrect specs.

In the meantime, here’s what I can tell you in a useful way about **what “UltraSlim Smart Toothbrush” models like this typically include**, and what to check on the exact listing:

## What it’s generally like (typical features)
Smart toothbrushes in this category usually offer:
- **Sonic oscillation** (high-frequency vibrations) for plaque removal
- **Slim/compact handle design** (“UltraSlim”)
- **Multiple brushing modes** (commonly: Clean / Sensitive / Whitening / Gum Care)
- **Timer + pressure guidance**
  - A **2-minute timer** (often with quadrant pacing)
  - **Pressure sensor** that alerts you if you brush too hard
- **Rechargeable battery** with a **charging base or dock**
- **Bluetooth/ap

<br>

This same example with an older OpenAI model ("gpt-3.5-turbo") was producing the following output:
```
The AeroGlide UltraSlim Smart Toothbrush by Boie is a high-tech toothbrush designed to provide a superior cleaning experience. It features a slim and sleek design that makes it easy to hold and maneuver in the mouth. The toothbrush is equipped with smart technology that tracks your brushing habits and provides real-time feedback to help you improve your oral hygiene routine.
```

<br>

Modern language models hallucinate significantly less often than earlier generations thanks to improvements in training, reasoning, and alignment. They are generally better at following instructions, acknowledging uncertainty, and avoiding unsupported claims. 

However, hallucinations have not been eliminated, especially when answering questions about niche topics, recent events, or information outside the model's knowledge. For this reason, it's still important to verify critical information using reliable sources.